# Workstream 1: Data Inventory And Schema Check

`EDA/eda-experiment-design.md` の Workstream 1 に対応します。dataset split、schema、ID coverage を再確認し、以後の診断で `all_tracks` catalog を推薦対象として固定します。

主な既存成果物: `EDA/tables/data_inventory.csv`, `EDA/tables/id_coverage.csv`, `EDA/summary/input-data-eda-summary.md`.


## Setup


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

try:
    from IPython.display import Image, Markdown, display
except Exception:
    Image = Markdown = display = None

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 160)


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "EDA").exists():
            return candidate
    raise RuntimeError("Repository root was not found from the current working directory.")

ROOT = find_repo_root(Path.cwd())
EDA_DIR = ROOT / "EDA"
TABLE_DIR = EDA_DIR / "tables"
FIGURE_DIR = EDA_DIR / "figures"
SUMMARY_DIR = EDA_DIR / "summary"
EXPERIMENT_DIR = ROOT / "mcrs" / "experiments"
INFERENCE_DIR = ROOT / "exp" / "inference"

for directory in (TABLE_DIR, FIGURE_DIR, SUMMARY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"TABLE_DIR={TABLE_DIR}")


In [ ]:
def read_table(name: str, **kwargs) -> pd.DataFrame:
    path = TABLE_DIR / name
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def read_csv_path(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def show_df(df: pd.DataFrame, n: int = 20) -> None:
    if df.empty:
        print("empty dataframe")
        return
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n).to_string(index=False))


def show_image(name: str) -> None:
    path = FIGURE_DIR / name
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return
    if display is not None and Image is not None:
        display(Image(filename=str(path)))
    else:
        print(path)


def save_table(df: pd.DataFrame, name: str) -> Path | None:
    if df.empty:
        print(f"skip empty table: {name}")
        return None
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print(f"saved: {path.relative_to(ROOT)} ({len(df):,} rows)")
    return path


def barplot(df: pd.DataFrame, *, x: str, y: str, hue: str | None = None, title: str = "", rotate: int = 0, figsize=(10, 4)) -> None:
    if df.empty:
        print("skip empty plot")
        return
    fig, ax = plt.subplots(figsize=figsize)
    if sns is not None:
        sns.barplot(data=df, x=x, y=y, hue=hue, ax=ax)
    else:
        df.plot(kind="bar", x=x, y=y, ax=ax)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=rotate)
    fig.tight_layout()
    plt.show()


## Inventory Tables


In [ ]:
data_inventory = read_table("data_inventory.csv")
id_coverage = read_table("id_coverage.csv")

show_df(data_inventory)
show_df(id_coverage)


## Row Count And Split Sanity Checks


In [ ]:
expected_rows = {
    ("TalkPlayData-Challenge-Dataset", "train"): 15199,
    ("TalkPlayData-Challenge-Dataset", "test"): 1000,
    ("TalkPlayData-Challenge-Blind-A", "test"): 80,
    ("TalkPlayData-Challenge-Track-Metadata", "all_tracks"): 47071,
    ("TalkPlayData-Challenge-Track-Metadata", "test_tracks"): 7405,
    ("TalkPlayData-Challenge-User-Metadata", "all_users"): 8772,
}

if data_inventory.empty:
    row_check = pd.DataFrame()
else:
    row_check = data_inventory.copy()
    row_check["expected_rows"] = row_check.apply(lambda r: expected_rows.get((r["dataset"], r["split"])), axis=1)
    row_check["row_delta"] = row_check["rows"] - row_check["expected_rows"]
    row_check = row_check[["dataset", "split", "rows", "expected_rows", "row_delta", "num_columns", "size_mb", "file"]]

show_df(row_check, 50)


## Coverage Gates


In [ ]:
if id_coverage.empty:
    gate_df = pd.DataFrame()
else:
    gate_df = id_coverage.copy()
    gate_df["status"] = np.where(gate_df["missing"].fillna(0).eq(0), "pass", "inspect")
    gate_df = gate_df.sort_values(["status", "missing"], ascending=[True, False])

show_df(gate_df, 50)

critical_checks = [
    "dev/test music track_id in all_tracks",
    "train music track_id in all_tracks",
    "test_tracks subset of all_tracks",
]
if not gate_df.empty:
    show_df(gate_df[gate_df["check"].isin(critical_checks)], 20)


## Visual Checks


In [ ]:
if not data_inventory.empty:
    plot_df = data_inventory.sort_values(["dataset", "split"])
    barplot(plot_df, x="split", y="rows", hue="dataset", title="Rows by dataset split", rotate=25, figsize=(12, 5))

if not id_coverage.empty:
    coverage_df = id_coverage.sort_values("coverage_rate")
    barplot(coverage_df, x="check", y="coverage_rate", title="ID coverage rate", rotate=80, figsize=(12, 5))


## Findings / Decisions / Next Actions

- Findings:
- Decisions:
- Next actions:
